# Modern Pipelines in Polars

The `.pipe()` method exists in Polars and works identically — but Polars' expression API makes many inline steps cleaner too.

## Setup

In [ ]:
import polars as pl
import numpy as np
from datetime import date, timedelta
import random

def make_ts_df() -> pl.DataFrame:
    start = date(2018, 1, 1)
    dates = [start + timedelta(days=i) for i in range(366)]
    values = [
        None if random.random() < 0.05 else np.random.normal(0, 1)
        for _ in range(366)
    ]
    return pl.DataFrame({"date": dates, "value": values})

date_df = make_ts_df()
date_df.head()

> **Note - difference to pandas:** Dates are already native `pl.Date` objects — no string parsing needed. Polars infers the type from Python `date` objects directly. The `parse_dates` step from the pandas version is therefore unnecessary.

## Inline approach (before pipelines)

Before analysing the data, it can be cleaned. Let's imagine it need to be cleaned in two ways:

- Clean the `null` values.
- Remove outliers.

In [ ]:
(
    date_df
    .drop_nulls()
    .filter(pl.col("value").is_between(-2.0, 2.0))
)

This can be done better.

If you were to just look at the code above it could be hard to understand what is going on. Also, with a new date_df dataframe, the same logic will need to be applied.

This is a very simple example, but imagine there were more cleaning steps required.

---

## Pipeline abstraction

The `.pipe()` method allows you to pass a function that accepts a DataFrame as its first argument. This is a very nice flow:

- It can be easily reused this pipeline (or parts of it) for different datasets.
- If there is ever a bug, this pipeline makes it easier to locate — since every step is merely a function, you'll know exactly where the process is breaking.
- Each function can be given a descriptive name, so the pipeline reads like a story.

In [ ]:
def remove_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """Removes rows with missing values"""
    return df.drop_nulls()

def remove_outliers(df: pl.DataFrame) -> pl.DataFrame:
    """Removes values less than -2 and greater than 2"""
    return df.filter(pl.col("value").is_between(-2.0, 2.0))


prep_df = (
    date_df
    .pipe(remove_nulls)
    .pipe(remove_outliers)
)
prep_df

In [ ]:
# You can inspect any step's docstring
help(remove_nulls)

---

## Pipeline abstraction on higher levels

`.pipe()` can also accept keyword arguments. This allows you to change threshold values from a high level without touching the original function — encouraging you to write general, reusable functions.

In [ ]:
def remove_outliers(df: pl.DataFrame, min_value: float = None, max_value: float = None) -> pl.DataFrame:
    """Removes outliers outside [min_value, max_value]"""
    if min_value is None or max_value is None:
        raise ValueError("You must provide both min_value and max_value")
    return df.filter(pl.col("value").is_between(min_value, max_value))


(
    date_df
    .pipe(remove_nulls)
    .pipe(remove_outliers, min_value=-2, max_value=2)
)

---

## Bonus: decorators

Because each pipeline step is a plain Python function, you get the full decorator toolkit for free. The decorator code is **identical** to the pandas version — decorators operate on the function, not the DataFrame type.

Here two decorators are added:
- `log_shape` — logs the DataFrame shape before and after each step
- `log_time` — logs how long each step takes

In [ ]:
from functools import wraps
import time


def log_shape(func):
    """Logs DataFrame shape before and after the pipeline step"""
    @wraps(func)
    def wrapper(df: pl.DataFrame, *args, **kwargs) -> pl.DataFrame:
        result = func(df, *args, **kwargs)
        print(f"{func.__name__} => before: {df.shape}  after: {result.shape}")
        return result
    return wrapper


def log_time(func):
    """Logs how long the pipeline step takes"""
    @wraps(func)
    def wrapper(df: pl.DataFrame, *args, **kwargs) -> pl.DataFrame:
        t = time.perf_counter()
        result = func(df, *args, **kwargs)
        print(f"{func.__name__}: {time.perf_counter() - t:.4f}s")
        return result
    return wrapper


@log_shape
@log_time
def remove_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """Removes rows with missing values"""
    return df.drop_nulls()


@log_shape
@log_time
def remove_outliers(df: pl.DataFrame, min_value: float = -2.0, max_value: float = 2.0) -> pl.DataFrame:
    """Removes outliers outside [min_value, max_value]"""
    return df.filter(pl.col("value").is_between(min_value, max_value))


prep_df = (
    date_df
    .pipe(remove_nulls)
    .pipe(remove_outliers, min_value=-1)
)

Benefits of a standard decorator on pipeline steps:

1. While writing code, it helps you discover what is happening — if rows disappear when they shouldn't, the log gives you a proxy.
2. When this code goes to production (e.g. Airflow), you get logging for free. If something goes wrong, you can debug more easily.

---

## Exercise: San Francisco crime data

Rewrite the following as a Polars pipeline:

```python
# original
(
    sanfran
    .rename({col: col.lower() for col in sanfran.columns})
    .rename({"dates": "date"})
    .with_columns(
        pl.col("date").str.to_datetime().dt.date().alias("date")
    )
    .sort("date")
    .filter(
        (pl.col("date") >= pl.date(2004, 1, 1)) &
        (pl.col("date") <= pl.date(2014, 12, 31))
    )
    .group_by_dynamic(
        "date",
        every="1mo",   # monthly buckets
        period="1mo",
        closed="left"
    )
    .agg(
        pl.count("category").alias("category")
    )
    .with_columns(
        pl.col("category")
        .rolling_mean(window_size=10)
        .alias("category_rolling")
    )
)
```

In [ ]:
sanfran = pl.read_csv("data/san_fran_crime_sample.csv", try_parse_dates=True)
sanfran.head()

In [ ]:
# %load answers/polars_pipeline.py

## Conclusion

> **"Pipelines are the correct way to write Polars."**

Even if you take this with a grain of salt, writing your code as named, testable, composable functions keeps your notebooks clear, your pipelines debuggable, and your colleagues happy.